In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install nltk rouge-score bitsandbytes

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 128.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 111.7 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl si

In [ ]:
!rm -rf /root/nltk_data/corpora/wordnet
!rm -rf /root/nltk_data/corpora/omw-1.4

In [ ]:
%cd /content/drive/MyDrive/연세_석사/졸업논문/실험/

/content/drive/MyDrive/연세_석사/졸업논문/실험


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/연세_석사/졸업논문/실험/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from exaone_pad_100_dataset import CustomDataset
from exaone_alpha_X_train_valid import train, valid, calculate_rouge, compute_bleu_rouge_meteor
from exaone_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt
from exaone_None_run import run

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    # 'model_name': 'MLP-KTLim/llama-3-Korean-Bllossom-8B',
    'model_name': 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct', # 패딩위치: 오른쪽
    'path': '/content/drive/MyDrive/연세_석사/졸업논문/실험/results/',
    'model_path': '6000_0.5_pad_O_EXAONE-3.5-2.4B-Instruct_best_cot_ver_{}.pt',
    'checkpoint_path': '6000_0.5_pad_O_EXAONE-3.5-2.4B-Instruct_cot_checkpoint_ver_{}.pt',
    'json_output_file' : '6000_0.5_pad_O_EXAONE-3.5-2.4B-Instruct_cot_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': '6000_0.5_pad_O_EXAONE-3.5-2.4B-Instruct_df_test_pred_cot_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'cot', # student, tot, cot, basic 확인!!
    'alpha' : 0.5,
    'epochs' :  7,
    'start_epoch': 6, # 끊겼을 때 다시 시작할 에폭
    'ver': 3,
    'ver_start': 2, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 3.012643443743388, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['ver_start'] == 0 and meta_data['start_epoch'] == 0:
    all_results = {}
else:
    with open(meta_data['path'] + meta_data['json_output_file'], 'r') as f:
        all_results = json.load(f)

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/{}_student_train_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/ab_train_data.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    if meta_data['start_epoch'] == 0:
        meta_data['save_val_loss'] = 100

    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver
    df = df.sample(frac = 1, random_state = ver).reset_index(drop = True)
    split_idx = int(meta_data['data_num'] * 0.9)
    df_train = df.iloc[:split_idx]
    df_valid = df.iloc[split_idx:int(meta_data['data_num'])]
    # df_train = df.iloc[:10]
    # df_valid = df.iloc[10:20]

    if str(ver) not in all_results.keys():
        all_results[str(ver)] = {
            'meta_data' : meta_data,
            'time': None,
            'results': []
        }
    start = time.time()
    all_results = run(ver, meta_data, df_train, df_valid, calculate_rouge, content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, all_results)
    all_results[str(ver)]['time'] = time.strftime('%X', time.localtime(time.time() - start))

    with open(meta_data['path'] + meta_data['json_output_file'], 'w') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=4)

    meta_data['start_epoch'] = 0

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


========== version 2 ==========


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/70.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.93M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`




여기

Epoch: 7
--------------------


  0%|          | 0/5400 [00:00<?, ?it/s]WARNING:transformers_modules.LGAI-EXAONE.EXAONE-3.5-2.4B-Instruct.e949c91dec92095908d34e6b560af77dd0c993f8.modeling_exaone:`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
[C_loss : 3.0121]: 100%|██████████| 600/600 [09:45<00:00,  1.02it/s]


best model was saved


In [ ]:
!rm -rf /root/nltk_data/corpora/wordnet
!rm -rf /root/nltk_data/corpora/omw-1.4

In [ ]:
# l4로 돌리기
import sys
sys.path.append('/content/drive/MyDrive/연세_석사/졸업논문/실험/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from exaone_pad_100_dataset import CustomDataset
from exaone_alpha_O_train_valid import train, valid, calculate_rouge
from exaone_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, test_prompt
from exaone_run import run
from exaone_test import pred_test

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    # 'model_name': 'MLP-KTLim/llama-3-Korean-Bllossom-8B',
    'model_name': 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct', # 패딩위치: 오른쪽
    'path': '/content/drive/MyDrive/연세_석사/졸업논문/실험/results/',
    'model_path': '6000_0.5_pad_O_EXAONE-3.5-2.4B-Instruct_best_cot_ver_{}.pt',
    'checkpoint_path': '6000_0.5_pad_O_EXAONE-3.5-2.4B-Instruct_cot_checkpoint_ver_{}.pt',
    'json_output_file' : '6000_0.5_pad_O_EXAONE-3.5-2.4B-Instruct_cot_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': '6000_0.5_pad_O_EXAONE-3.5-2.4B-Instruct_df_test_pred_cot_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'cot', # student, tot, cot, basic 확인!!
    'alpha' : 0.5,
    'epochs' :  7,
    'start_epoch': 0, # 끊겼을 때 다시 시작할 에폭
    'ver': 3,
    'ver_start': 0, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 100, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df_test = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/{}_student_test_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df_test = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/ab_test_data.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver

    # test_df = pred_test(ver, meta_data, df_test, calculate_rouge, tot_prompt1, tot_prompt2, tot_prompt3, tot_prompt4, cot_prompt1, cot_prompt2, basic_prompt)
    test_df = pred_test(ver, meta_data, df_test, content2, test_prompt)

test_df
print(test_df['prediction'][0])

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# alpha만 없는 버전

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install nltk rouge-score bitsandbytes

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 130.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 106.0 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl siz

In [3]:
!rm -rf /root/nltk_data/corpora/wordnet
!rm -rf /root/nltk_data/corpora/omw-1.4
%cd /content/drive/MyDrive/연세_석사/졸업논문/실험/

/content/drive/MyDrive/연세_석사/졸업논문/실험


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/연세_석사/졸업논문/실험/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from exaone_pad_100_dataset import CustomDataset
from exaone_alpha_X_train_valid import train, valid, calculate_rouge, compute_bleu_rouge_meteor
from exaone_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt
from exaone_None_run import run

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    'model_name': 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct', # 패딩위치: 오른쪽
    'path': '/content/drive/MyDrive/연세_석사/졸업논문/실험/results/',
    'model_path': 'real_6000_None_pad_O_EXAONE-3.5-2.4B-Instruct_best_cot_ver_{}.pt',
    'checkpoint_path': 'real_6000_None_pad_O_EXAONE-3.5-2.4B-Instruct_cot_checkpoint_ver_{}.pt',
    'json_output_file' : 'real_6000_None_pad_O_EXAONE-3.5-2.4B-Instruct_cot_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': 'real_6000_None_pad_O_EXAONE-3.5-2.4B-Instruct_df_test_pred_cot_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'cot', # student, tot, cot, basic 확인!!
    'alpha' : None,
    'epochs' :  7,
    'start_epoch': 3, # 끊겼을 때 다시 시작할 에폭
    'ver': 3,
    'ver_start': 2, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 6.0788081574440005, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['ver_start'] == 0 and meta_data['start_epoch'] == 0:
    all_results = {}
else:
    with open(meta_data['path'] + meta_data['json_output_file'], 'r') as f:
        all_results = json.load(f)

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/{}_student_train_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/ab_train_data.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    if meta_data['start_epoch'] == 0:
        meta_data['save_val_loss'] = 100

    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver
    df = df.sample(frac = 1, random_state = ver).reset_index(drop = True)
    split_idx = int(meta_data['data_num'] * 0.9)
    df_train = df.iloc[:split_idx]
    df_valid = df.iloc[split_idx:int(meta_data['data_num'])]
    # df_train = df.iloc[:10]
    # df_valid = df.iloc[10:20]

    if str(ver) not in all_results.keys():
        all_results[str(ver)] = {
            'meta_data' : meta_data,
            'time': None,
            'results': []
        }
    start = time.time()
    all_results = run(ver, meta_data, df_train, df_valid, calculate_rouge, content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, all_results)
    all_results[str(ver)]['time'] = time.strftime('%X', time.localtime(time.time() - start))

    with open(meta_data['path'] + meta_data['json_output_file'], 'w') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=4)

    meta_data['start_epoch'] = 0

In [ ]:
!rm -rf /root/nltk_data/corpora/wordnet
!rm -rf /root/nltk_data/corpora/omw-1.4
%cd /content/drive/MyDrive/연세_석사/졸업논문/실험/

/content/drive/MyDrive/연세_석사/졸업논문/실험


In [4]:
# l4로 돌리기
import sys
sys.path.append('/content/drive/MyDrive/연세_석사/졸업논문/실험/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from exaone_pad_100_dataset import CustomDataset
from exaone_alpha_X_train_valid import train, valid, calculate_rouge, compute_bleu_rouge_meteor
from exaone_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, test_prompt
from exaone_None_run import run
from exaone_test import pred_test

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    'model_name': 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct', # 패딩위치: 오른쪽
    'path': '/content/drive/MyDrive/연세_석사/졸업논문/실험/results/',
    'model_path': 'real_6000_None_pad_O_EXAONE-3.5-2.4B-Instruct_best_cot_ver_{}.pt',
    'checkpoint_path': 'real_6000_None_pad_O_EXAONE-3.5-2.4B-Instruct_cot_checkpoint_ver_{}.pt',
    'json_output_file' : 'real_6000_None_pad_O_EXAONE-3.5-2.4B-Instruct_cot_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': 'real_6000_None_pad_O_EXAONE-3.5-2.4B-Instruct_df_test_pred_cot_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'cot', # student, tot, cot, basic 확인!!
    'alpha' : None,
    'epochs' :  7,
    'start_epoch': 0, # 끊겼을 때 다시 시작할 에폭
    'ver': 3,
    'ver_start': 2, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 100, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df_test = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/{}_student_test_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df_test = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/ab_test_data.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver

    # test_df = pred_test(ver, meta_data, df_test, calculate_rouge, tot_prompt1, tot_prompt2, tot_prompt3, tot_prompt4, cot_prompt1, cot_prompt2, basic_prompt)
    test_df = pred_test(ver, meta_data, df_test, content2, test_prompt)

test_df
print(test_df['prediction'][0])

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


========== version 2 ==========


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/70.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.93M [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/4.65G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
100%|██████████| 1918/1918 [2:52:46<00:00,  5.40s/it]

당신은 화자의 발언에 공감하는 청자 역할입니다.
화자의 발화에 대한 공감 답변을 생성하세요.
### 화자의 발화: 엄마, 이번에 주전 선수의 대리로 경기에 출전하게 됐어요! 항상 보결로 벤치에 앉아 모두를 지켜보기만 했었는걸요. 직접 뛸 수 있게 돼서 너무나 기뻐요!
### 청자의 공감 답변:
정말 자랑스러워 보이네요! 그동안 응원했던 시간들이 떠오르고 정말 뿌듯할 것 같아요. 연습하고 준비한 덕분일 거예니까 자신감 갖고 즐겁게 임해 주세요. 경기에서 좋은 모습 보여주면 저도 무척 행복할게요. 어떤 플레이가 기대되나요?


In [ ]:
# 다 없는 버전

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install nltk rouge-score bitsandbytes

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 118.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 104.6 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl siz

In [ ]:
!rm -rf /root/nltk_data/corpora/wordnet
!rm -rf /root/nltk_data/corpora/omw-1.4

In [ ]:
%cd /content/drive/MyDrive/연세_석사/졸업논문/실험/

/content/drive/MyDrive/연세_석사/졸업논문/실험


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/연세_석사/졸업논문/실험/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from exaone_pad_X_dataset import CustomDataset
from exaone_alpha_X_train_valid import train, valid, calculate_rouge, compute_bleu_rouge_meteor
from exaone_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt
from exaone_None_run import run

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    # 'model_name': 'MLP-KTLim/llama-3-Korean-Bllossom-8B',
    'model_name': 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct', # 패딩위치: 오른쪽
    'path': '/content/drive/MyDrive/연세_석사/졸업논문/실험/results/',
    'model_path': '6000_None_pad_X_EXAONE-3.5-2.4B-Instruct_best_cot_ver_{}.pt',
    'checkpoint_path': '6000_None_pad_X_EXAONE-3.5-2.4B-Instruct_cot_checkpoint_ver_{}.pt',
    'json_output_file' : '6000_None_pad_X_EXAONE-3.5-2.4B-Instruct_cot_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': '6000_None_pad_X_EXAONE-3.5-2.4B-Instruct_df_test_pred_cot_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'cot', # student, tot, cot, basic 확인!!
    'alpha' : None,
    'epochs' :  7,
    'start_epoch': 6, # 끊겼을 때 다시 시작할 에폭
    'ver': 1,
    'ver_start': 0, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 6.780230226516724, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['ver_start'] == 0 and meta_data['start_epoch'] == 0:
    all_results = {}
else:
    with open(meta_data['path'] + meta_data['json_output_file'], 'r') as f:
        all_results = json.load(f)

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/{}_student_train_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/ab_train_data.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    if meta_data['start_epoch'] == 0:
        meta_data['save_val_loss'] = 100

    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver
    df = df.sample(frac = 1, random_state = ver).reset_index(drop = True)
    split_idx = int(meta_data['data_num'] * 0.9)
    df_train = df.iloc[:split_idx]
    df_valid = df.iloc[split_idx:int(meta_data['data_num'])]
    # df_train = df.iloc[:10]
    # df_valid = df.iloc[10:20]

    if str(ver) not in all_results.keys():
        all_results[str(ver)] = {
            'meta_data' : meta_data,
            'time': None,
            'results': []
        }
    start = time.time()
    all_results = run(ver, meta_data, df_train, df_valid, calculate_rouge, content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, all_results)
    all_results[str(ver)]['time'] = time.strftime('%X', time.localtime(time.time() - start))

    with open(meta_data['path'] + meta_data['json_output_file'], 'w') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=4)

    meta_data['start_epoch'] = 0

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


========== version 0 ==========


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/70.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.93M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`




여기

Epoch: 7
--------------------


  0%|          | 0/5400 [00:00<?, ?it/s]WARNING:transformers_modules.LGAI-EXAONE.EXAONE-3.5-2.4B-Instruct.e949c91dec92095908d34e6b560af77dd0c993f8.modeling_exaone:`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
[C_loss : 6.7792]: 100%|██████████| 600/600 [09:24<00:00,  1.06it/s]


best model was saved


In [ ]:
!rm -rf /root/nltk_data/corpora/wordnet
!rm -rf /root/nltk_data/corpora/omw-1.4

In [ ]:
# l4로 돌리기
import sys
sys.path.append('/content/drive/MyDrive/연세_석사/졸업논문/실험/')

import numpy as np
import pandas as pd
import os, re, math, random, time, json, pickle, gc, ast
from tqdm import tqdm
from collections import defaultdict
import itertools as it

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

# from transformers import AdamW
from transformers.optimization import get_cosine_schedule_with_warmup
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import get_peft_config, get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

from rouge_score import rouge_scorer

import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import single_meteor_score

from exaone_pad_X_dataset import CustomDataset
from exaone_alpha_X_train_valid import train, valid, calculate_rouge, compute_bleu_rouge_meteor
from exaone_prompt import content1, content2, tot_prompt1, tot_prompt2, tot_prompt3, cot_prompt, basic_prompt, test_prompt
from exaone_None_run import run
from exaone_test import pred_test

device = torch.device(f"cuda" if torch.cuda.is_available() else "cpu")
# !!!!!!!!!!!!start_epoch, save_val_loss 꼭 바꾸기!!!!!!!!!!!
meta_data = {
    # 'model_name': 'MLP-KTLim/llama-3-Korean-Bllossom-8B',
    'model_name': 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct', # 패딩위치: 오른쪽
    'path': '/content/drive/MyDrive/연세_석사/졸업논문/실험/results/',
    'model_path': '6000_None_pad_X_EXAONE-3.5-2.4B-Instruct_best_cot_ver_{}.pt',
    'checkpoint_path': '6000_None_pad_X_EXAONE-3.5-2.4B-Instruct_cot_checkpoint_ver_{}.pt',
    'json_output_file' : '6000_None_pad_X_EXAONE-3.5-2.4B-Instruct_cot_training_results.json', # tot, cot, basic 확인!!
    'test_file_path': '6000_None_pad_X_EXAONE-3.5-2.4B-Instruct_df_test_pred_cot_best_model_ver_{}.csv', # tot, cot, basic 확인!!
    'mode' : 'cot', # student, tot, cot, basic 확인!!
    'alpha' : None,
    'epochs' :  7,
    'start_epoch': 0, # 끊겼을 때 다시 시작할 에폭
    'ver': 1,
    'ver_start': 0, # 끊겼을 때 다시 시작할 버전
    'batch_size' : 1,
    'weight_decay' : 1e-6,
    'max_len' : 420,
    'start_lr' : 2e-5,
    'data_num': 6000,
    'save_val_loss': 100, # 끊겼을 때 다시 시작할 loss 상한값 # !!!!!!!!!! 바꾸기!!!!!!!!!!
    'device' : str(device)
}

def set_seeds(seed = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if meta_data['mode'] == 'cot' or meta_data['mode'] == 'tot':
    df_test = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/{}_student_test_df.csv'.format(meta_data['mode']), encoding = 'utf-8')
else:
    df_test = pd.read_csv('/content/drive/MyDrive/연세_석사/졸업논문/실험/ab_test_data.csv', encoding = 'utf-8')

for ver in range(meta_data['ver']):
    if ver < meta_data['ver_start']:
        continue
    print('=' * 10, f'version {ver}', '=' * 10)
    set_seeds(ver)
    meta_data['seed'] = ver
    meta_data['ver_start'] = ver
    ver_start = ver

    # test_df = pred_test(ver, meta_data, df_test, calculate_rouge, tot_prompt1, tot_prompt2, tot_prompt3, tot_prompt4, cot_prompt1, cot_prompt2, basic_prompt)
    test_df = pred_test(ver, meta_data, df_test, content2, test_prompt)

test_df
print(test_df['prediction'][0])

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


========== version 0 ==========
The repository for LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 1/1918 [00:05<3:01:27,  5.68s/it]

['당신은 화자의 발언에 공감하는 청자 역할입니다.\n화자의 발화에 대한 공감 답변을 생성하세요.\n### 화자의 발화: 엄마, 이번에 주전 선수의 대리로 경기에 출전하게 됐어요! 항상 보결로 벤치에 앉아 모두를 지켜보기만 했었는걸요. 직접 뛸 수 있게 돼서 너무나 기뻐요!\n### 청자의 공감 답변:\n형도 많이 기대할 거야. 응원해줘요! 우리 팀이 더 강해질 것 같아 보여. 힘내라, 잘 해낼 거지? 계속 응원하고, 너의 열정적인 모습도 격려하고 싶어. 새로운 기회가 되길 바라며... 그 경험 덕분에 더 좋은 플레이 가능해질지도 모른다니? 화이팅해요!']


...

['당신은 화자의 발언에 공감하는 청자 역할입니다.\n화자의 발화에 대한 공감 답변을 생성하세요.\n### 화자의 발화: 사람들하고 같이 다니다 보면 맛집이라고 소개하는 경우가 많잖아? 각 sns에 올라오는 맛집 정보도 허다하고 말야. 근데 난 소개한 맛집 가보고 난 슬프게도 내 입맛에 맞은 적이 거의 없어.\n### 청자의 공감 답변:\n, 맛있음으로 시작했던 나도 경험에서 배운 거지! 사실 맛집이라 해서 꼭 그렇게 잘 먹었을지도 모르니까, 당신이랑 다른 맛집 찾아서 새로운 곳 탐험해보는 건 어때? 그럼 새로운 추억 생길 거라고 생각돼! 함께 가봐야겠네.']


100%|█████████▉| 1917/1918 [2:45:29<00:05,  5.20s/it]

['당신은 화자의 발언에 공감하는 청자 역할입니다.\n화자의 발화에 대한 공감 답변을 생성하세요.\n### 화자의 발화: 언니 나 너무나도 충격적인 소식을 들었어. 마음이 진정이 안 되는데 어떡하지?\n### 청자의 공감 답변:\n화자가 그렇게 힘든 상황에 처했다는 게 안타깝구나. 너도 많은 고민과 걱정이 있겠지만, 필요한 도움을 얻을 수 있도록 주변 사람들께 이야기해보는 건 어때?) 그동안 많이 힘들었겠지만, 같이 이겨낼 방법 찾아줄게! 괜찮아질 때까지 내 옆에서 응원할 테니까 너무 끙끙 앓지 말고 조금씩 노력하다 보면 좋겠다구. 언제든지 이야기 나누고 싶으면 얘기해 줘.']


100%|██████████| 1918/1918 [2:45:37<00:00,  5.18s/it]

['당신은 화자의 발언에 공감하는 청자 역할입니다.\n화자의 발화에 대한 공감 답변을 생성하세요.\n### 화자의 발화: 언니 방금 길거리를 지나가다가 전에 내가 키우던 반려견 있잖아? 닮은 강아지를 마주쳤어.\n### 청자의 공감 답변:\n, 정말 멋진 일이네! 비슷한 친구를 찾다니 얼마나 기분 좋았을까 싶다. 혹시 그 강아지와 어떻게 만났는지 이야기해줄래? 나도 언젠가 그런 경험이나 새로운 친구들과 만나면 더 넓은 세상을 만날 수 있지 않아? 그리고 항상 네 행복함만 생각해줘서 감사하단 말 꼭 전하고 왔니? 함께 즐거운 추억 만들었으면 해, 엄마야. \n이렇게 반응하면 상대방의 감정과 경험을 이해하고 응원하며 대화할 준비되어 있음을 보여주는 것이다.']
당신은 화자의 발언에 공감하는 청자 역할입니다.
화자의 발화에 대한 공감 답변을 생성하세요.
### 화자의 발화: 엄마, 이번에 주전 선수의 대리로 경기에 출전하게 됐어요! 항상 보결로 벤치에 앉아 모두를 지켜보기만 했었는걸요. 직접 뛸 수 있게 돼서 너무나 기뻐요!
### 청자의 공감 답변:
형도 많이 기대할 거야. 응원해줘요! 우리 팀이 더 강해질 것 같아 보여. 힘내라, 잘 해낼 거지? 계속 응원하고, 너의 열정적인 모습도 격려하고 싶어. 새로운 기회가 되길 바라며... 그 경험 덕분에 더 좋은 플레이 가능해질지도 모른다니? 화이팅해요!
